# IronStore Enterprise AI Assistant — v1 (Pinecone + semantic search)

**Goal:** employees ask a question in plain language → the assistant finds the right internal document chunks → answers **only from them**, with sources, and says so when it doesn't know.

| Step | What happens |
|---|---|
| 0 | Speed fix for this PC |
| 1 | Settings, keys, models |
| 2 | Look at the documents |
| 3 | Load the PDFs (+ metadata: area, page, last reviewed, outdated?) |
| 4 | Split into chunks (with a small context header) |
| 5 | Create the Pinecone index |
| 6 | Embed + upload the chunks (only when needed) |
| 7 | One semantic search, to see what Pinecone returns |
| 8 | Multi-query: 3 extra versions of the question |
| 9 | Search all versions → **deduplicate** the chunks |
| 10 | Answer with sources (links to the PDF page) |
| 11 | Test questions |
| 12 | Findings + ideas for v2 |

**Choices for v1:** Pinecone (serverless, cosine), OpenAI `text-embedding-3-small` (semantic only, no keyword search),
`gpt-4o-mini` to rewrite questions (cheap and fast), `gpt-4.1-mini` to write the answer (more careful). That's already a small *model routing*: a cheap model for the easy job, a stronger one for the important job.


## Step 0 — Speed fix (this PC only)
Try IPv4 before IPv6, so each API call takes ~1 s instead of ~40 s. Safe to run twice.

In [1]:
import socket

if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

## Step 1 — Settings, keys and models
The keys come from environment variables (`OPENAI_API_KEY`, `PINECONE_API_KEY`) or a `.env` file — they are never printed.

In [2]:
import os
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()

DOCS_DIR = Path(r"C:\Users\Windows\Desktop\Ironhack\04_llm\project-brief-enterprise-ai-assistant\documents\extracted\documents")
INDEX_NAME = "ironstore-assistant"
EMBED_MODEL = "text-embedding-3-small"   # 1536 numbers per chunk
EMBED_DIM = 1536
REWRITE_MODEL = "gpt-4o-mini"            # writes the extra question versions
ANSWER_MODEL = "gpt-4.1-mini"            # writes the final answer

print("OpenAI key found:  ", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key found:", bool(os.getenv("PINECONE_API_KEY")))
print("Documents folder:  ", DOCS_DIR.exists())

OpenAI key found:   True
Pinecone key found: True
Documents folder:   True


## Step 2 — Look at the documents
Two kinds of files: **internal policies** (one folder per department) and **product catalogs**.
One catalog is from **2021** — the brief warns about *outdated* information, so we will mark it.

In [3]:
pdfs = sorted(p for p in DOCS_DIR.rglob("*.pdf"))
overview = pd.DataFrame({
    "area": [p.parent.name for p in pdfs],
    "file": [p.name for p in pdfs],
})
print(len(pdfs), "PDF files")
overview.groupby("area").size().rename("files").to_frame()

83 PDF files


,files
area,
Customer_support,3
Deliveries,3
Finance,3
HR_policies,3
IT,2
Legal_Terms,3
Sales,3
product_catalog,63


## Step 3 — Load the PDFs (one record per document)
For each PDF we keep the text of every page and some **metadata** read from the first lines:
- `area` (folder), `title`, `doc_id`, `version`, `last_reviewed`
- `outdated = True` when the document was last reviewed before 2026 (→ the 2021 catalog)

The pages are glued together into one text, but we remember where each page starts, so every chunk can still say *which page* it comes from.

In [4]:
import re
import pymupdf

def read_pdf(path: Path) -> dict:
    pages = [page.get_text() for page in pymupdf.open(path)]
    text = "".join(pages)
    head = text[:600]

    def find(pattern):
        match = re.search(pattern, head)
        return match.group(1) if match else ""

    last_reviewed = find(r"Last reviewed(?: in|:)?\s*([A-Z][a-z]+ \d{4})")
    year = int(last_reviewed[-4:]) if last_reviewed else 0
    if path.parent.name == "product_catalog":
        title = path.stem.replace("IronStore_", "").replace("_", " ")
    else:
        title = head.strip().splitlines()[0]
    page_starts, pos = [], 0
    for page in pages:
        page_starts.append(pos)
        pos += len(page)
    return {
        "path": path,
        "file": path.name,
        "area": path.parent.name,
        "doc_type": "catalog" if path.parent.name == "product_catalog" else "policy",
        "title": title,
        "doc_id": find(r"(?:Policy ID|Document ID|Policy reference):\s*(\S+)"),
        "version": find(r"Version:\s*([\d.]+)"),
        "last_reviewed": last_reviewed or "unknown",
        "outdated": 0 < year < 2026,
        "text": text,
        "page_starts": page_starts,
        "pages": len(pages),
    }

t0 = time.time()
docs = [read_pdf(p) for p in pdfs]
print(f"{len(docs)} documents, {sum(d['pages'] for d in docs)} pages, "
      f"{sum(len(d['text']) for d in docs):,} characters in {time.time() - t0:.1f} s")
pd.DataFrame(docs)[["area", "title", "doc_id", "version", "last_reviewed", "outdated", "pages"]].head(25)

83 documents, 2308 pages, 2,436,798 characters in 2.5 s


,area,title,doc_id,version,last_reviewed,outdated,pages
0,Customer_support,Customer Support Escalation Procedures,CS-POL-014,3.2,August 2026,False,7
1,Customer_support,Customer Support Knowledge Base Style Guide,CS-KB-STD-004,,August 2026,False,7
2,Customer_support,Refund & Complaint Handling Policy,CS-POL-014,3.2,August 2026,False,7
3,Deliveries,Returns & Reverse Logistics Procedure,DEL-RET-004,3.2,August 2026,False,6
4,Deliveries,Shipping & Delivery SLA Policy,DEL-SLA-004,,August 2026,False,6
5,Deliveries,Warehouse Operations Manual,DEL-WH-001,3.2,August 2026,False,7
6,Finance,Budget Approval Workflow,FIN-PR-014,,August 2026,False,6
7,Finance,IronStore Expense Reimbursement Policy,FIN-EXP-004,,August 2026,False,6
8,Finance,Payroll Processing Procedures,FIN-PAY-004,3.2,August 2026,False,6
9,HR_policies,Employee Handbook,HR-001,,August 2026,False,7


**What we find in the data** (the brief said there would be inconsistencies):

In [5]:
meta = pd.DataFrame(docs)
print("Outdated documents:", meta.loc[meta.outdated, "file"].tolist())
dup_ids = meta[meta.doc_id != ""].groupby("doc_id").file.apply(list)
print("Same document ID used twice:", dup_ids[dup_ids.str.len() > 1].to_dict())

Outdated documents: ['IronStore_Old_Product_Catalog_2021.pdf']
Same document ID used twice: {'CS-POL-014': ['IronStore_Customer_Support_Escalation_Procedures.pdf', 'IronStore_Refund_and_Complaint_Handling_Policy.pdf']}


## Step 4 — Split into chunks
- `RecursiveCharacterTextSplitter` counted in **tokens** (`chunk_size=450`, `chunk_overlap=60`): big enough for a policy paragraph or ~2 products, small enough to stay precise.
- We look up where each chunk starts in the document text → its **page**. (`add_start_index=True` alone gives wrong positions here, because the overlap is counted in tokens but searched in characters.)
- Every chunk gets a one-line **header** (`[title | area | page | last reviewed | OUTDATED]`).
  The header is embedded too, so a question like *"remote work policy"* also matches chunks whose text never says "remote work policy".

In [6]:
from bisect import bisect_right
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=450, chunk_overlap=60
)

chunks = []
for doc in docs:
    pieces = splitter.split_documents([Document(page_content=doc["text"])])
    start = 0
    for n, piece in enumerate(pieces):
        start = max(doc["text"].find(piece.page_content, start), start)  # where this chunk begins
        page = bisect_right(doc["page_starts"], start)                    # → page 1, 2, 3 …
        header = (f"[{doc['title']} | {doc['area']} | page {page} | last reviewed {doc['last_reviewed']}"
                  + (" | OUTDATED" if doc["outdated"] else "") + "]")
        chunks.append({
            "id": f"{doc['path'].stem}-{n:03d}",
            "text": header + "\n" + piece.page_content,
            "source": doc["file"],
            "title": doc["title"],
            "area": doc["area"],
            "doc_type": doc["doc_type"],
            "page": page,
            "last_reviewed": doc["last_reviewed"],
            "outdated": doc["outdated"],
            "path": str(doc["path"]),
        })

lengths = pd.Series([len(c["text"]) for c in chunks])
print(len(chunks), "chunks | characters per chunk: median", int(lengths.median()), "max", lengths.max())
print("\n--- example chunk ---\n" + chunks[200]["text"][:900])

1759 chunks | characters per chunk: median 1654 max 2365

--- example chunk ---
[Product Catalog Audio and Headphones | product_catalog | page 19 | last reviewed August 2026]
•  2.5-inch colour display with adjustable brightness
•  Dual alarm with weekday and weekend settings
•  Mains powered with backup clock battery
•  Sleep timer and kitchen timer
•  3.5 mm headphone output
•  Rated output: 5 W RMS
Availability: In stock
Warehouse location: IT-MIL, radio shelf R-05
30. Sonetra Emergency Radio E1
SKU: IS-AUD-RAD-0030
Category: Audio & Headphones / Emergency and outdoor radios
Short description: Portable FM/AM radio with rechargeable battery, hand crank, solar panel, and emergency light.
Price: €44.95
Key specifications:
•  FM and AM reception
•  Manual hand-crank and solar charging
•  2,000 mAh internal battery
•  USB-A emergency power output
•  1 W speaker and 3.5 mm headphone output
•  Integrated LED torch and reading lamp
•  Water-resistant casing rated IPX3
• 


## Step 5 — Create the Pinecone index
Serverless index on AWS `us-east-1` (free tier region), **cosine** similarity, 1536 dimensions (= size of `text-embedding-3-small` vectors).
Created only if it doesn't exist yet.

In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("index created")
index = pc.Index(host=pc.describe_index(INDEX_NAME).host)
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=1759, metric='cosine', namespaces=1)


## Step 6 — Embed and upload the chunks
- One OpenAI call per 1000 chunks turns each chunk into a vector (cost for all ~2.4 M characters: about **1–2 cents**).
- Upload = `id` + `values` (the vector) + `metadata` (text, file, page, …).
- **Skipped** when Pinecone already holds exactly this number of chunks → re-running the notebook is fast. Set `REBUILD = True` after changing the chunking.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
REBUILD = False

already = index.describe_index_stats().total_vector_count
if REBUILD or already != len(chunks):
    if already:
        index.delete(delete_all=True)
    t0 = time.time()
    vectors = embeddings.embed_documents([c["text"] for c in chunks])
    print(f"embedded {len(vectors)} chunks in {time.time() - t0:.0f} s")
    t0 = time.time()
    index.upsert(
        vectors=[{"id": c["id"], "values": v, "metadata": c} for c, v in zip(chunks, vectors)],
        batch_size=100, show_progress=False,
    )
    print(f"uploaded in {time.time() - t0:.0f} s")
    while index.describe_index_stats().total_vector_count < len(chunks):
        time.sleep(2)
else:
    print("Pinecone already has all", already, "chunks — nothing to upload")
print(index.describe_index_stats())

Pinecone already has all 1759 chunks — nothing to upload
DescribeIndexStatsResponse(dimension=1536, total_vector_count=1759, metric='cosine', namespaces=1)


## Step 7 — One semantic search
The question becomes a vector; Pinecone returns the `top_k` chunks whose vectors point in the most similar direction.
**score** = cosine similarity (1.0 = same meaning). No keywords are involved.

In [9]:
def search(vector, k=6):
    result = index.query(vector=vector, top_k=k, include_metadata=True)
    return [{"id": m.id, "score": round(m.score, 3), **m.metadata} for m in result.matches]

question = "What is our remote work policy?"
hits = search(embeddings.embed_query(question))
pd.DataFrame(hits)[["score", "title", "page", "id"]]

,score,title,page,id
0,0.719,Remote Work Policy,6,IronStore_Remote_Work_Policy-006
1,0.717,Remote Work Policy,5,IronStore_Remote_Work_Policy-005
2,0.699,Remote Work Policy,2,IronStore_Remote_Work_Policy-001
3,0.696,Remote Work Policy,4,IronStore_Remote_Work_Policy-004
4,0.684,Remote Work Policy,1,IronStore_Remote_Work_Policy-000
5,0.680,Remote Work Policy,2,IronStore_Remote_Work_Policy-002


## Step 8 — Multi-query: 3 more versions of the question
Employees phrase things differently from the documents ("work from home" vs. "remote-first / hybrid").
A cheap model writes **3 alternative versions**; we search with all **4**. More versions → more chances to hit the right chunk.
`with_structured_output` makes the model return a clean Python list instead of free text.

In [10]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class Rewrites(BaseModel):
    questions: list[str] = Field(description="3 alternative versions of the question")

rewrite_prompt = ChatPromptTemplate.from_template(
    "You help search IronStore's internal documents (HR, IT, finance, legal, sales, customer support, "
    "deliveries, product catalogs).\n"
    "Write 3 different versions of the employee's question. Use other words and synonyms an official "
    "policy or catalog would use. Keep product names, numbers and SKUs unchanged.\n\nQuestion: {question}"
)
rewriter = rewrite_prompt | ChatOpenAI(model=REWRITE_MODEL, temperature=0.3).with_structured_output(Rewrites)

def make_queries(question):
    return [question] + rewriter.invoke({"question": question}).questions[:3]

queries = make_queries(question)
for i, q in enumerate(queries):
    print(f"Q{i}: {q}")

Q0: What is our remote work policy?
Q1: What are the guidelines regarding telecommuting within our organization?
Q2: Can you provide details on the company's policy for remote work arrangements?
Q3: What are the stipulations concerning our remote work procedures?


## Step 9 — Search all versions, then deduplicate
Similar questions find **many of the same chunks**. Before the LLM sees them we:
1. embed all 4 questions in **one** call and search Pinecone **in parallel** (faster);
2. **dedup by chunk id** — keep each chunk once, with its best score, and count how many questions found it (`hits`: found by several versions = more trustworthy);
3. **dedup by content** — drop chunks with identical text under a different id;
4. keep the best `MAX_CHUNKS`.

The table below shows the overlap: ✓ = this question version found this chunk.

In [11]:
from concurrent.futures import ThreadPoolExecutor

MAX_CHUNKS = 8

def retrieve(question, k=6, verbose=False):
    queries = make_queries(question)
    vectors = embeddings.embed_documents(queries)
    with ThreadPoolExecutor(max_workers=len(vectors)) as pool:
        results = list(pool.map(lambda v: search(v, k), vectors))

    raw = [{**hit, "query": f"Q{i}"} for i, hits in enumerate(results) for hit in hits]

    best = {}
    for hit in raw:                                   # 2. dedup by id
        if hit["id"] not in best:
            best[hit["id"]] = {**hit, "hits": 0}
        best[hit["id"]]["hits"] += 1
        best[hit["id"]]["score"] = max(best[hit["id"]]["score"], hit["score"])

    unique, seen_text = [], set()                     # 3. dedup by content
    for hit in sorted(best.values(), key=lambda h: (-h["hits"], -h["score"])):
        body = " ".join(hit["text"].split("\n", 1)[-1].split()).lower()
        if body not in seen_text:
            seen_text.add(body)
            unique.append(hit)
    unique = unique[:MAX_CHUNKS]                      # 4. keep the best

    if verbose:
        for i, q in enumerate(queries):
            print(f"Q{i}: {q}")
        print(f"\n{len(raw)} chunks found in total → {len(best)} after id dedup → "
              f"{len(unique)} kept for the LLM\n")
        overlap = pd.crosstab([pd.Series([h["id"] for h in raw], name="chunk")],
                              pd.Series([h["query"] for h in raw], name="found by"))
        display(overlap.replace({1: "✓", 0: ""}))
    return unique

kept = retrieve("Can I work from home, and how many days do I need to be in the office?", verbose=True)
pd.DataFrame(kept)[["hits", "score", "title", "page", "id"]]

Q0: Can I work from home, and how many days do I need to be in the office?
Q1: Am I permitted to telecommute, and what is the required number of days for in-office attendance?
Q2: Is remote work an option for me, and how many days am I obligated to be present at the workplace?
Q3: Can I engage in work-from-home arrangements, and what is the minimum number of days I must report to the office?

24 chunks found in total → 8 after id dedup → 8 kept for the LLM



found by,Q0,Q1,Q2,Q3
chunk,,,,
IronStore_Employee_Handbook-001,,,,✓
IronStore_Employee_Handbook-002,,✓,,
IronStore_Remote_Work_Policy-000,✓,,✓,
IronStore_Remote_Work_Policy-001,✓,✓,✓,✓
IronStore_Remote_Work_Policy-002,✓,✓,✓,✓
IronStore_Remote_Work_Policy-004,✓,✓,✓,✓
IronStore_Remote_Work_Policy-005,✓,✓,✓,✓
IronStore_Remote_Work_Policy-006,✓,✓,✓,✓


,hits,score,title,page,id
0,4,0.630,Remote Work Policy,2,IronStore_Remote_Work_Policy-001
1,4,0.622,Remote Work Policy,2,IronStore_Remote_Work_Policy-002
2,4,0.602,Remote Work Policy,5,IronStore_Remote_Work_Policy-005
3,4,0.569,Remote Work Policy,4,IronStore_Remote_Work_Policy-004
4,4,0.542,Remote Work Policy,6,IronStore_Remote_Work_Policy-006
5,2,0.526,Remote Work Policy,1,IronStore_Remote_Work_Policy-000
6,1,0.473,Employee Handbook,2,IronStore_Employee_Handbook-002
7,1,0.470,Employee Handbook,1,IronStore_Employee_Handbook-001


## Step 10 — Answer with sources
Each kept chunk becomes a numbered source `[S1]`, `[S2]`, … The prompt tells the model to:
- answer **only from the sources** and cite them;
- say clearly when the sources don't contain the answer (no guessing);
- prefer the most recently reviewed document and **warn** when something only comes from the OUTDATED 2021 catalog.

Below the answer we list the sources with a **link that opens the PDF at the right page** (bonus point of the brief).

In [12]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the IronStore internal assistant for employees.\n"
     "Answer ONLY with information from the sources below. Cite them inline like [S1] or [S2][S3].\n"
     "If the sources do not contain the answer, say: \"I couldn't find this in the internal documents\" "
     "and suggest which department could help. Never invent numbers, names or rules.\n"
     "If sources disagree, prefer the most recently reviewed one and mention the difference.\n"
     "Sources marked OUTDATED (the 2021 catalog) must not be used for current prices or stock; "
     "if they are the only match, say the information is outdated.\n"
     "Be concise; use bullet points for procedures. Answer in the language of the question.\n\n"
     "Sources:\n{context}"),
    ("human", "{question}"),
])
answer_llm = ChatOpenAI(model=ANSWER_MODEL, temperature=0)
answer_chain = answer_prompt | answer_llm

def ask(question, verbose=False):
    t0 = time.time()
    kept = retrieve(question, verbose=verbose)
    t1 = time.time()
    context = "\n\n".join(f"[S{i}] {c['text']}" for i, c in enumerate(kept, 1))
    answer = answer_chain.invoke({"context": context, "question": question}).content
    t2 = time.time()

    cited = sorted({int(n) for n in re.findall(r"\[S(\d+)\]", answer)})
    lines = [f"- **[S{n}]** {kept[n-1]['title']}, page {kept[n-1]['page']}"
             f" — [open PDF]({Path(kept[n-1]['path']).as_uri()}#page={kept[n-1]['page']})"
             + (" ⚠️ outdated" if kept[n-1]["outdated"] else "")
             for n in cited if n <= len(kept)]
    display(Markdown(f"**Q: {question}**\n\n{answer}\n\n**Sources**\n" + ("\n".join(lines) or "- none")
                     + f"\n\n<sub>retrieval {t1 - t0:.1f} s · answer {t2 - t1:.1f} s · "
                       f"best score {max(c['score'] for c in kept):.2f}</sub>"))
    return {"answer": answer, "sources": [kept[n-1] for n in cited if n <= len(kept)]}

_ = ask("Can I work from home, and how many days do I need to be in the office?")

**Q: Can I work from home, and how many days do I need to be in the office?**

Yes, you can work from home if your role is eligible and approved for remote or hybrid work after your probation period. The process requires submitting a request through Workday, manager assessment, People Partner review, and country-specific approval if needed [S1].

Regarding office attendance:
- Hybrid employees are generally expected to attend the office about two days per week.
- Teams may require more office days during peak periods or major projects, with at least five working days' notice from managers [S1].
- Remote-first employees mainly work remotely but must attend team events, planning sessions, training, or other business needs [S1].

Your specific working model and office attendance requirements will be confirmed by your hiring manager and People Partner and recorded in Workday [S1].

**Sources**
- **[S1]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)

<sub>retrieval 1.4 s · answer 1.8 s · best score 0.61</sub>

## Step 11 — Test questions
A mix of policies, products, an **outdated-catalog trap**, and a question that is **not** in the documents.

In [14]:
_ = ask("How many days a week I can work from home?")

**Q: How many days a week I can work from home?**

- If you have a hybrid working model, you are generally expected to attend the IronStore office an average of two days per week, working remotely on the other days.
- The exact number of remote days may vary depending on team requirements, peak periods, or projects.
- Your working model and attendance expectations are confirmed by your hiring manager and People Partner and recorded in Workday.
- You must attend designated office days and required business events.
- Remote work must be from an approved location within your employing country and meet policy requirements [S1][S3].

**Sources**
- **[S1]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)
- **[S3]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)

<sub>retrieval 2.4 s · answer 1.4 s · best score 0.65</sub>

In [15]:
_ = ask("A customer wants to return an item. How many days do they have?")

**Q: A customer wants to return an item. How many days do they have?**

Customers have 30 calendar days from delivery to return unopened and resalable goods, unless a longer legal or commercial period applies. Some product-specific exceptions include personalised goods (normally non-returnable unless faulty) and sealed hygiene or beauty products (non-returnable after seal removal where permitted by law) [S4].

**Sources**
- **[S4]** Refund & Complaint Handling Policy, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/Customer_support/IronStore_Refund_and_Complaint_Handling_Policy.pdf#page=1)

<sub>retrieval 2.0 s · answer 1.9 s · best score 0.58</sub>

In [16]:
_ = ask("How much does the Nordview 55-inch QLED TV cost and is it in stock?")

**Q: How much does the Nordview 55-inch QLED TV cost and is it in stock?**

The Nordview NQ-55U7 55-inch 4K QLED Smart TV costs €649.00 and is in stock with 18 units available at the Rotterdam DC, Aisle E12, Bin 04 [S1].

**Sources**
- **[S1]** Product Catalog Electronics, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Electronics.pdf#page=1)

<sub>retrieval 2.1 s · answer 1.1 s · best score 0.58</sub>

In [17]:
_ = ask("What is the price of the IronStore 24 inch Full HD office monitor?")

**Q: What is the price of the IronStore 24 inch Full HD office monitor?**

The IronStore 24" Full HD Office Monitor is priced at €149.99 according to the 2021 catalog [S1]. However, this information is outdated, and the current catalog does not list this product. For the latest price, please check with the Electronics department.

**Sources**
- **[S1]** Old Product Catalog 2021, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Old_Product_Catalog_2021.pdf#page=1) ⚠️ outdated

<sub>retrieval 2.0 s · answer 1.1 s · best score 0.56</sub>

In [18]:
_ = ask("What is the CEO's annual salary?")

**Q: What is the CEO's annual salary?**

I couldn't find this in the internal documents. You may contact the People Operations or Finance department for information on executive salaries.

**Sources**
- none

<sub>retrieval 1.4 s · answer 1.0 s · best score 0.34</sub>

A question with only a product **code** (SKU) — hard for pure semantic search:

In [19]:
_ = ask("What is SKU PRJ-PCB-PB1080 and how much does it cost?")

**Q: What is SKU PRJ-PCB-PB1080 and how much does it cost?**

I couldn't find this SKU PRJ-PCB-PB1080 in the internal documents. You may want to contact the Project Management or Procurement department for more information.

**Sources**
- none

<sub>retrieval 1.6 s · answer 1.3 s · best score 0.42</sub>

## Step 12 — Findings and ideas for v2

**✅ What works well**
- Policy questions land on the right document (remote work → Remote Work Policy, time off → Leave & Absence Policy, returns → Refund policy) with correct facts: 2 office days/week, 10 / 20 days notice, 30-day returns.
- Product lookup by name works (Nordview TV → €649, 18 in stock, warehouse bin).
- **Outdated trap handled:** the 24" monitor only exists in the 2021 catalog → the answer gives the price *and* warns that it is outdated.
- **"I don't know" works:** the CEO salary question gets no invented answer (best score only ~0.34 vs ~0.55–0.65 for real hits).
- **Dedup:** 4 question versions × 6 results = 24 hits → 8 unique chunks; the chunks found by all 4 versions are the most relevant ones.
- Fast enough: ~1.5–2 s retrieval + ~1–2 s answer. Indexing all 83 PDFs: ~25 s, ~1–2 cents.

**⚠️ What is not great yet**
- Catalog chunks can cut a product in half (fixed token size) → a product's price and stock may land in two chunks.
- **Pure semantic search fails on exact codes:** asking for SKU `PRJ-PCB-PB1080` returns "couldn't find", although the projector is in the Electronics catalog. The vectors know the *meaning*, not the exact string.
- Two different policies share the ID `CS-POL-014` (data inconsistency) — the assistant can't tell them apart by ID.
- The multi-query step costs one extra LLM call (~0.7 s) on every question, even easy ones.

**🚀 Ideas for v2 (in order of value)**
1. **Product-aware chunking**: one chunk per product (split at each `SKU:` block) + metadata `sku`, `price`, `category`.
2. **Hybrid search** (semantic + keyword/sparse vectors in Pinecone) for SKUs, IDs and exact names.
3. **Metadata filters / routing**: send product questions only to `doc_type = catalog` (and skip `outdated`), policy questions only to policies.
4. **Reranker** (e.g. Pinecone `bge-reranker-v2-m3`) on the 8 deduplicated chunks → better order, fewer chunks to the LLM.
5. **Score threshold**: if the best score < ~0.40, answer "not found" without calling the LLM (faster, cheaper).
6. **Chat memory** for follow-up questions, then the API (FastAPI) + frontend + deploy.
